# 5 - A Full Example

The code of the tutorial page "5 - A Full Example": the seawater desalination plant that ships with Acquirium, with one year of data.

Before each run, erase the stored data and create the example files:

    acquirium clear
    acquirium example init --force

In [1]:
import polars as pl

import acquirium as aq

acq = aq.init()

## 1. Insert the plant model

In [2]:
acq.insert_graph_file("example/model.ttl", source_id="plant")

## 2. Register the streams of the file

The file is the source, and every column other than `time` is one stream.

In [3]:
data = pl.read_csv("example/data.csv", try_parse_dates=True)
stream_names = [column for column in data.columns if column != "time"]

acq.register_streams([
    {"source_id": "csv-data", "stream_name": name} for name in stream_names
])

Insert the data, one list of `(timestamp, value)` pairs per stream.

In [4]:
times = data["time"].to_list()

acq.insert_timeseries_batch("csv-data", {
    name: list(zip(times, data[name].to_list())) for name in stream_names
})

{'ok': True, 'rows_inserted': 280320, 'batches': 6}

## 3. Find what is not connected

The properties of the model that have no stream:

In [5]:
acq.query().property(has_stream=False)

property
str
"""wbs:P1-efficiency"""
"""wbs:PXR-brine-out-flow-mass-wa…"
"""wbs:conn-cartridge-filtration-…"
"""wbs:RO-in-flow-mass-water"""
"""wbs:RO-in-temperature-k"""


The streams that are not connected to a property of the model:

In [6]:
acq.query().measurement().where(is_disconnected=True)

data,data.label
str,str
"""acq:c81d82c6-46f8-5396-a488-c7…","""intake-in-flow-rate-m3s"""
"""acq:40701b11-c32d-5b45-98aa-21…","""P2-mechanical-power"""
"""acq:0a4ab90c-af5c-5fb6-aff2-69…","""conn-cartridge-filtration-to-S…"
"""acq:22b1bf04-14cc-5a84-a947-f7…","""conn-cartridge-filtration-to-S…"
"""acq:5d7ecdb5-062f-5af3-afbe-96…","""RO-in-flow-mass-water"""


## 4. Connect the streams to the model

The mapping below says which property of the model each stream belongs to.

In [7]:
mapping = {
    "intake-in-flow-rate-m3s":                         "wbs:intake-in-flow-rate-m3s",
    "intake-in-tds-concentration":                     "wbs:intake-in-tds-concentration",
    "intake-in-tss-concentration-kgm3":                "wbs:intake-in-tss-concentration-kgm3",
    "conn-cartridge-filtration-to-S1-temperature-k":   "wbs:conn-cartridge-filtration-to-S1-temperature-k",
    "conn-cartridge-filtration-to-S1-flow-mass-water": "wbs:conn-cartridge-filtration-to-S1-flow-mass-water",
    "conn-cartridge-filtration-to-S1-flow-mass-tds":   "wbs:conn-cartridge-filtration-to-S1-flow-mass-tds",
    "P1-mechanical-power":                             "wbs:P1-mechanical-power",
    "P2-mechanical-power":                             "wbs:P2-mechanical-power",
    "RO-in-temperature-k":                             "wbs:RO-in-temperature-k",
    "RO-in-flow-mass-water":                           "wbs:RO-in-flow-mass-water",
    "RO-in-flow-mass-tds":                             "wbs:RO-in-flow-mass-tds",
    "RO-out-flow-mass-water":                          "wbs:RO-out-flow-mass-water",
    "RO-out-flow-mass-tds":                            "wbs:RO-out-flow-mass-tds",
    "RO-out-retentate-pressure-pa":                    "wbs:RO-out-retentate-pressure-pa",
    "RO-out-retentate-flow-mass-water":                "wbs:RO-out-retentate-flow-mass-water",
    "RO-out-retentate-flow-mass-tds":                  "wbs:RO-out-retentate-flow-mass-tds",
    "PXR-brine-out-flow-mass-water":                   "wbs:PXR-brine-out-flow-mass-water",
    "PXR-brine-out-flow-mass-tds":                     "wbs:PXR-brine-out-flow-mass-tds",
    "storage-tank-3-out-flow-rate-m3s":                "wbs:storage-tank-3-out-flow-rate-m3s",
    "storage-tank-3-out-tds-concentration":            "wbs:storage-tank-3-out-tds-concentration",
    "conn-cartridge-filtration-to-S1-pressure-pa":     "wbs:conn-cartridge-filtration-to-S1-pressure-pa",
    "P1-efficiency":                                   "wbs:P1-efficiency",
    "P1-out-pressure-pa":                              "wbs:P1-out-pressure-pa",
    "P2-efficiency":                                   "wbs:P2-efficiency",
    "PXR-efficiency":                                  "wbs:PXR-efficiency",
    "RO-membrane-area":                                "wbs:RO-membrane-area",
    "RO-in-pressure-pa":                               "wbs:RO-in-pressure-pa",
    "RO-out-pressure-pa":                              "wbs:RO-out-pressure-pa",
    "PXR-brine-out-pressure-pa":                       "wbs:PXR-brine-out-pressure-pa",
    "ferric-chloride-addition-ferric-chloride-dose":   "wbs:ferric-chloride-addition-ferric-chloride-dose",
    "anti-scalant-addition-anti-scalant-dose":         "wbs:anti-scalant-addition-anti-scalant-dose",
    "lime-addition-lime-dose":                         "wbs:lime-addition-lime-dose",
}

matches = [
    {"source_id": "csv-data", "stream_name": name, "property_uri": prop}
    for name, prop in mapping.items()
]

acq.register_streams(matches)

Both lists are now empty.

In [8]:
acq.query().property(has_stream=False)

Query
  property: property, without a stream

In [9]:
acq.query().measurement().where(is_disconnected=True)

Query
  data: measurement, is_disconnected=True

## 5. Query the plant

The pumps of the plant:

In [10]:
acq.query().entity("pump").include("label").metadata()

pump,pump.label
str,str
"""wbs:intake""","""Seawater Onshore Intake"""
"""wbs:P2""","""Booster Pump P2"""
"""wbs:P1""","""High-pressure Pump P1"""


The reverse osmosis membrane downstream of each pump:

In [11]:
acq.query().entity("pump").related("Reverse Osmosis Membrane", direction="downstream").metadata()

pump,Reverse Osmosis Membrane
str,str
"""wbs:P1""","""wbs:RO"""
"""wbs:P2""","""wbs:RO"""


The power streams of the pumps:

In [12]:
q = acq.query().entity("pump").measurement(quantity_kind="power")

q

pump,pump_data,pump_data.label
str,str,str
"""wbs:P2""","""wbs:P2-mechanical-power""","""P2 mechanical power"""
"""wbs:P1""","""wbs:P1-mechanical-power""","""P1 mechanical power"""


The nearest pressure stream upstream of the membrane:

In [13]:
ro = acq.query().entity(uri="wbs:RO", alias="ro")

ro.measurement(direction="upstream", quantity_kind="pressure").metadata()

ro,ro_upstream_entity,ro_upstream_data,ro_upstream_data.label
str,str,str,str
"""wbs:RO""","""wbs:RO-in""","""wbs:RO-in-pressure-pa""","""RO in pressure pa"""


The pump power of the first three hours of July:

In [14]:
q.dataframe(
    start="2025-07-01",
    end="2025-07-01 03:00",
)

time,pump_data__P1 mechanical power,pump_data__P2 mechanical power
"datetime[μs, UTC]",f64,f64
2025-07-01 00:00:00 UTC,1.0658e6,108456.097041
2025-07-01 01:00:00 UTC,1.0913e6,116735.330076
2025-07-01 02:00:00 UTC,1.0295e6,96421.121065
2025-07-01 03:00:00 UTC,1.0421e6,103118.66318


The same streams in kilowatts:

In [15]:
q.data().convert_to("kW").dataframe(limit=3)

time,pump_data__P1 mechanical power,pump_data__P2 mechanical power
"datetime[μs, UTC]",f64,f64
2025-01-01 00:00:00 UTC,1187.411568,106.089859
2025-01-01 01:00:00 UTC,1181.608011,101.473944
2025-01-01 02:00:00 UTC,1165.345244,97.750261


Stop Acquirium.

In [16]:
aq.shutdown()